<a href="https://colab.research.google.com/github/juergenlandauer/FoundationModelsArchaeology/blob/main/05_object_detection_YOLOv12_DINOv3_V05_longer_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLOv12 + DINOv3 Object Detection for Archaeological Sites

This notebook uses **YOLOv12 enhanced with DINOv3** (self-supervised Vision Transformer features) for detecting archaeological sites in satellite/LiDAR imagery.

Based on the [DINOV3-YOLOV12](https://github.com/Sompote/DINOV3-YOLOV12) repository, which integrates Meta's DINOv2/v3 features into the YOLOv12 detection pipeline.

**Workflow:**
1. Download site/nonsite image patches
2. Convert to YOLO detection format (centered bounding boxes on sites, nonsites as background)
3. Train YOLOv12+DINOv3 model
4. Evaluate on test set
5. Run sliding-window inference on large tiles

In [1]:
#PREPROCESSING_METHOD = 'none'  # Options: 'clahe3', 'hillshade', 'none'
PREPROCESSING_METHOD = 'clahe3'
#PREPROCESSING_METHOD = 'hillshade'

## Install Dependencies

In [2]:
!pip uninstall -y ultralytics opencv-python-headless 2>/dev/null
!pip install -q git+https://github.com/Sompote/DINOV3-YOLOV12.git
!pip install -Uq transformers timm supervision
!pip install -Uq albumentations opencv-python-headless==4.10.0.84
!pip install -Uq rvt-py rasterio geopandas

Found existing installation: opencv-python-headless 4.13.0.92
Uninstalling opencv-python-headless-4.13.0.92:
  Successfully uninstalled opencv-python-headless-4.13.0.92
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 112.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 6.8 MB/s eta 0:00:00


## Dataset

In [3]:
USE_RAW = True
#USE_RAW = False

### Use the hillshaded hillfort data

In [4]:
if not USE_RAW: # England Hillshade
    INPUT_ZIP_URL_NONSITES = 'https://www.dropbox.com/scl/fi/tkar49txfsaz8w7xcfweu/England_Negatives_for_DINOv2.zip?rlkey=8t7n4abxdkqkc6m4j7r5hbgns&dl=0'
    INPUT_ZIP_URL_SITES    = 'https://www.dropbox.com/scl/fi/l4c16t64uyeijg2ccqpya/England_Positives_for_DINOv2.zip?rlkey=tpqrj16ywkm0nuhplm6kn2d8p&dl=0'

### Use the raw data

In [5]:
if USE_RAW:
    # Khmer TIF
    INPUT_ZIP_URL_NONSITES = 'https://www.dropbox.com/scl/fi/q0mddolngvou51vyojhii/NonsitesKhmer.zip?rlkey=d718q7vonuh1g07yhajn4a8h0&dl=0'
    INPUT_ZIP_URL_SITES    = 'https://www.dropbox.com/scl/fi/qhrby4cmb3rtgpptplbrl/TemplesKhmerTIF.zip?rlkey=0xxj479p68hoqxe8rlpsxvh8k&dl=0'

In [6]:
if USE_RAW:
    # India TIF
    INPUT_ZIP_URL_NONSITES = 'https://www.dropbox.com/scl/fi/rt1z7o3xsnv9e6nie91ke/500_random_satellite_patches.zip?rlkey=60j3lk3lyomn2vujpe8s4mcqk&st=sjd2offo&dl=0'
    INPUT_ZIP_URL_SITES    = 'https://www.dropbox.com/scl/fi/hkbok5uigif2hey4cxn22/positives.zip?rlkey=056g1t1oz3l233d8btatxjslh&dl=0'

In [17]:
if USE_RAW:
    # Castles TIF
    INPUT_ZIP_URL_NONSITES = 'https://www.dropbox.com/scl/fi/8px493jr03mgi0660tsfv/NonCastlesBingImages_768_19_V2.zip?rlkey=lr89m75x40o7sg6iikje0b2q3&st=3geldkd5&dl=0'
    # all
    INPUT_ZIP_URL_SITES    = 'https://www.dropbox.com/scl/fi/vm0ya2ddb0ly8eyrsqws0/BingImages_768_19_orig.zip?rlkey=8t2gte5ssiyus3svmdxxnko0y&dl=0'
    # labels
    INPUT_ZIP_URL_LABELS   = 'https://www.dropbox.com/scl/fi/l34uuyv4910nvuk5b4rq2/stricterYoloLabels.zip?rlkey=m26ln3t8myrru23evnvsyojpk&st=cxv09d9k&dl=0'
    # external test set (site images with separate labels)
    INPUT_ZIP_URL_TEST     = 'https://www.dropbox.com/scl/fi/q03s00xcxpu5lq3f2ii62/BingImages_768_19_orig_strict.zip?rlkey=swb81qpyj835g2upuy2zpbe9e&st=hyub5xf7&dl=0'

In [8]:
if USE_RAW:
    !rm -rf input output testset file.zip
    !mkdir -p input/nonsites input/sites
    !wget -O file.zip "$INPUT_ZIP_URL_NONSITES"
    !unzip -q file.zip -d input/nonsites
    !wget -O file.zip "$INPUT_ZIP_URL_SITES"
    !unzip -q file.zip -d input/sites

--2026-02-23 09:45:35--  https://www.dropbox.com/scl/fi/8px493jr03mgi0660tsfv/NonCastlesBingImages_768_19_V2.zip?rlkey=lr89m75x40o7sg6iikje0b2q3&st=3geldkd5&dl=0
Resolving www.dropbox.com (www.dropbox.com)... 162.125.81.18, 2620:100:6031:18::a27d:5112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.81.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://ucf2b26ee3dbcf8aaf9c9f46a9a1.dl.dropboxusercontent.com/cd/0/inline/C7dhS6K6UKKqLSO7Z9ZGaUj6UVvQ_jIY-rpQ0gJaCVrKdGQ-CVbnqjP7XGVEuDN82nKN37aMIhaf6aYMgXsweFsyHRlOv_OnECjuIJR2t2sMCXHSzvJ8ow1eXX0SU0zdZOs/file# [following]
--2026-02-23 09:45:36--  https://ucf2b26ee3dbcf8aaf9c9f46a9a1.dl.dropboxusercontent.com/cd/0/inline/C7dhS6K6UKKqLSO7Z9ZGaUj6UVvQ_jIY-rpQ0gJaCVrKdGQ-CVbnqjP7XGVEuDN82nKN37aMIhaf6aYMgXsweFsyHRlOv_OnECjuIJR2t2sMCXHSzvJ8ow1eXX0SU0zdZOs/file
Resolving ucf2b26ee3dbcf8aaf9c9f46a9a1.dl.dropboxusercontent.com (ucf2b26ee3dbcf8aaf9c9f46a9a1.dl.dropboxusercontent.com)... 162.125.81.15, 2620:

In [9]:
# Count files
!ls input/sites | wc -l
!ls input/nonsites | wc -l

379
1000


## Image Preprocessing Functions

In [10]:
import cv2 as cv
import numpy as np
import rvt.vis

clahe = cv.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))


def remove_outliers(image: np.ndarray):
    if np.isnan(image).sum() > 0:
        assert False
    min_zero = image[image > -10].min()
    image[image <= -10] = min_zero
    mymax = np.max(image)
    mymin = np.min(image)
    if mymax - mymin > 1000:
        print("outlier removal:", mymax)
        hi = np.percentile(image, 99).min()
        image[image > 1000 + mymin] = hi
    return image


def preprocess_hillshade(image: np.ndarray) -> np.ndarray:
    if image is None:
        return image
    img = np.asarray(image)
    if img.ndim == 3:
        img = img[:, :, 0]
    hillshade = rvt.vis.hillshade(
        dem=img.astype(np.float32),
        resolution_x=1.0, resolution_y=1.0,
        sun_azimuth=315, sun_elevation=35,
    )
    hs = ((hillshade - hillshade.min()) / (hillshade.max() - hillshade.min()) * 255).astype(np.uint8)
    return np.stack([hs, hs, hs], axis=-1)


def preprocess_none(image: np.ndarray) -> np.ndarray:
    if image is None:
        return image
    img = np.asarray(image)
    if img.dtype != np.uint8:
        if img.max() <= 1.0:
            img = (img * 255).astype(np.uint8)
        else:
            img = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
    if img.ndim == 2:
        return np.stack([img, img, img], axis=-1)
    elif img.ndim == 3 and img.shape[2] == 3:
        return img
    else:
        return np.stack([img[:, :, 0]] * 3, axis=-1)


def preprocess_clahe3(image: np.ndarray) -> np.ndarray:
    """Apply CLAHE via LAB color space (L channel only)."""
    img = np.asarray(image)

    # Normalise to uint8
    if img.dtype != np.uint8:
        img = remove_outliers(img.astype(np.float64))
        mymin, mymax = img.min(), img.max()
        img -= mymin
        if abs(mymax - mymin) > 0.05:
            img = img / (mymax - mymin)
        img = (img * 255).astype(np.uint8)

    # Ensure 3-channel RGB
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)
    elif img.shape[2] != 3:
        img = np.stack([img[:, :, 0]] * 3, axis=-1)

    # Apply CLAHE on L channel in LAB space
    lab = cv.cvtColor(img, cv.COLOR_RGB2LAB)
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv.cvtColor(lab, cv.COLOR_LAB2RGB)


PREPROCESSING_FUNCTIONS = {
    'clahe3': preprocess_clahe3,
    'hillshade': preprocess_hillshade,
    'none': preprocess_none,
}


def get_preprocessing_function():
    return PREPROCESSING_FUNCTIONS[PREPROCESSING_METHOD]


## Convert Classification Patches to YOLO Detection Format

**For positive samples (sites):**
- Uses provided YOLO annotations from the labels zip file when available
- Falls back to auto-generated centered bounding boxes (60% of image) if label not found

**For negative samples (nonsites):**
- Empty label files (background/no objects)

YOLO format: each image has a corresponding `.txt` label file with lines:
```
class_id center_x center_y width height
```
(all values normalized to 0–1)

In [11]:
# ── Dataset split configuration ──────────────────────────────────────
TRAIN_SPLIT = 0.8  # 65% for training
VAL_SPLIT = 0.2    # 25% for validation
TEST_SPLIT = 0.0   # 10% for testing

DATA_PERCENTAGE = 1.0  # Use 100% of available data (set to 0.5 to use only 50%, etc.)

In [12]:
import os
import glob
import random
import shutil
import yaml
import zipfile
import urllib.request
from PIL import Image
from sklearn.model_selection import train_test_split


BBOX_FRACTION = 0.6  # bbox covers central 60% of each site patch
DATASET_DIR = "detection_dataset_yolo"


def read_and_preprocess(path):
    """Read an image file, apply preprocessing, return RGB numpy array."""
    img = cv.imread(path, cv.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f"Could not read {path}")
    if len(img.shape) == 3 and img.shape[2] > 3:
        img = img[:, :, :3]
    if len(img.shape) == 3 and img.shape[2] == 3:
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    elif len(img.shape) == 2:
        img = cv.cvtColor(img, cv.COLOR_GRAY2RGB)
    img = get_preprocessing_function()(img)
    return img


def download_and_extract_labels(labels_url, extract_dir="input/yolo_labels"):
    """Download and extract YOLO labels from Dropbox zip file."""
    os.makedirs(extract_dir, exist_ok=True)

    # Convert Dropbox URL to direct download link
    if 'dropbox.com' in labels_url:
        download_url = labels_url.replace('www.dropbox.com', 'dl.dropboxusercontent.com')
        download_url = download_url.replace('?dl=0', '?dl=1')
        if '?dl=1' not in download_url:
            download_url = download_url + '?dl=1'
    else:
        download_url = labels_url

    zip_path = os.path.join(extract_dir, "labels.zip")
    print(f"Downloading YOLO labels from: {labels_url}")
    urllib.request.urlretrieve(download_url, zip_path)
    print(f"Downloaded to: {zip_path}")

    # Extract zip file
    extract_subdir = os.path.join(extract_dir, "extracted")
    os.makedirs(extract_subdir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_subdir)

    # Build a mapping: image_filename (without extension) -> label content
    labels_dict = {}
    for root, dirs, files in os.walk(extract_subdir):
        for file in files:
            if file.endswith('.txt'):
                label_path = os.path.join(root, file)
                with open(label_path, 'r') as f:
                    label_content = f.read()
                # Key is the filename without extension
                key = os.path.splitext(file)[0]
                labels_dict[key] = label_content

    print(f"Loaded {len(labels_dict)} label files")
    return labels_dict


def build_yolo_dataset(
    site_dir="input/sites",
    nonsite_dir="input/nonsites",
    output_dir=DATASET_DIR,
    train_split=TRAIN_SPLIT,
    val_split=VAL_SPLIT,
    test_split=TEST_SPLIT,
    data_percentage=DATA_PERCENTAGE,
    bbox_fraction=BBOX_FRACTION,
    max_nonsites_ratio=1.0,
    labels_dict=None,
    max_train_images=None, # New parameter to cap training set size
):
    """
    Convert classification patches to YOLO detection dataset.

    Creates the directory structure:
        output_dir/
            images/{train,valid,test}/
            labels/{train,valid,test}/
            data.yaml

    Parameters:
        train_split: Fraction of data for training (default: 0.65 = 65%)
        val_split: Fraction of data for validation (default: 0.25 = 25%)
        test_split: Fraction of data for testing (default: 0.10 = 10%)
        data_percentage: Fraction of total data to use (default: 1.0 = 100%)
        Note: train_split + val_split + test_split should equal 1.0

    If labels_dict is provided, ONLY uses site images that have a corresponding label.
    All other site images are excluded. Nonsites always get empty labels (background).

    When test_split=0.0, no test split is created from training data.
    Use setup_external_testset() to populate the test set from a separate source.
    """
    # Validate split percentages
    total_split = train_split + val_split + test_split
    if not (0.99 <= total_split <= 1.01):  # Allow small floating point errors
        raise ValueError(
            f"train_split ({train_split}) + val_split ({val_split}) + "
            f"test_split ({test_split}) must sum to 1.0, got {total_split}"
        )

    print(f"Dataset split: Train={train_split*100:.0f}%, Val={val_split*100:.0f}%, Test={test_split*100:.0f}%")
    if data_percentage < 1.0:
        print(f"Using {data_percentage*100:.0f}% of available data")

    # Collect image paths
    site_paths = sorted(glob.glob(os.path.join(site_dir, "**/*"), recursive=True))
    site_paths = [p for p in site_paths if os.path.isfile(p)]
    nonsite_paths = sorted(glob.glob(os.path.join(nonsite_dir, "**/*"), recursive=True))
    nonsite_paths = [p for p in nonsite_paths if os.path.isfile(p)]

    # Exclude the first 10 labels and images
    site_paths = site_paths[10:]
    ### QUICKHACK!!!!!!!!!!!!!!!!!!!!!

    print(f"Found {len(site_paths)} site patches, {len(nonsite_paths)} nonsite patches")

    # Apply data_percentage to reduce dataset size if specified
    if data_percentage < 1.0:
        random.seed(42)  # Ensure reproducibility
        n_sites_to_use = int(len(site_paths) * data_percentage)
        n_nonsites_to_use = int(len(nonsite_paths) * data_percentage)
        site_paths = random.sample(site_paths, max(1, n_sites_to_use))
        nonsite_paths = random.sample(nonsite_paths, max(1, n_nonsites_to_use))
        print(f"After applying {data_percentage*100:.0f}% data limit: {len(site_paths)} site patches, {len(nonsite_paths)} nonsite patches")

    # If labels_dict is provided, filter to only include labeled sites
    if labels_dict is not None:
        labeled_site_paths = []
        for path in site_paths:
            image_basename = os.path.splitext(os.path.basename(path))[0]
            if image_basename in labels_dict:
                labeled_site_paths.append(path)

        n_excluded = len(site_paths) - len(labeled_site_paths)
        print(f"Using labels_dict: Keeping {len(labeled_site_paths)} labeled sites, excluding {n_excluded} unlabeled sites")
        site_paths = labeled_site_paths

    # Cap nonsites to avoid extreme class imbalance in backgrounds
    max_nonsites = int(len(site_paths) * max_nonsites_ratio)
    if len(nonsite_paths) > max_nonsites:
        random.seed(42)
        nonsite_paths = random.sample(nonsite_paths, max_nonsites)
        print(f"Capped nonsites to {max_nonsites}")

    # Split into train/val/test using the specified percentages
    if test_split > 0:
        # First split: separate training from (validation + test)
        site_train, site_temp = train_test_split(site_paths, test_size=val_split + test_split, random_state=42)
        # Second split: separate validation from test
        relative_test = test_split / (val_split + test_split)
        site_val, site_test = train_test_split(site_temp, test_size=relative_test, random_state=42)

        ns_train, ns_temp = train_test_split(nonsite_paths, test_size=val_split + test_split, random_state=42)
        ns_val, ns_test = train_test_split(ns_temp, test_size=relative_test, random_state=42)

        splits = {
            "train": (site_train, ns_train),
            "valid": (site_val, ns_val),
            "test": (site_test, ns_test),
        }
    else:
        # No test split — all data goes to train/val
        actual_val_split = val_split / (train_split + val_split)
        site_train, site_val = train_test_split(site_paths, test_size=actual_val_split, random_state=42)
        ns_train, ns_val = train_test_split(nonsite_paths, test_size=actual_val_split, random_state=42)
        splits = {
            "train": (site_train, ns_train),
            "valid": (site_val, ns_val),
        }

    # Cap the number of training samples if max_train_images is specified
    if max_train_images is not None and "train" in splits:
        current_site_train, current_ns_train = splits["train"]

        all_train_paths = current_site_train + current_ns_train
        random.seed(42) # Ensure reproducibility for sampling
        random.shuffle(all_train_paths) # Shuffle to get a random subset

        if len(all_train_paths) > max_train_images:
            print(f"Capping total training images from {len(all_train_paths)} to {max_train_images} as requested.")
            sampled_train_paths = random.sample(all_train_paths, max_train_images)

            # Re-distribute into sites and nonsites based on sampled paths
            new_site_train = [p for p in sampled_train_paths if p in set(current_site_train)]
            new_ns_train = [p for p in sampled_train_paths if p in set(current_ns_train)]

            splits["train"] = (new_site_train, new_ns_train)
            print(f"New training split: {len(new_site_train)} sites, {len(new_ns_train)} background.")


    # Always create test directories (may be populated by setup_external_testset)
    os.makedirs(os.path.join(output_dir, "images", "test"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", "test"), exist_ok=True)

    for split_name, (sites, nonsites) in splits.items():
        img_dir = os.path.join(output_dir, "images", split_name)
        lbl_dir = os.path.join(output_dir, "labels", split_name)
        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(lbl_dir, exist_ok=True)

        image_id = 0
        n_sites = 0
        n_bg = 0

        # Process site patches (with provided labels)
        for path in sites:
            try:
                img = read_and_preprocess(path)
            except Exception as e:
                print(f"Skipping {path}: {e}")
                continue

            fname = f"{image_id:06d}.jpg"
            Image.fromarray(img).save(os.path.join(img_dir, fname), quality=95)

            # Get label from labels_dict
            image_basename = os.path.splitext(os.path.basename(path))[0]
            if labels_dict and image_basename in labels_dict:
                label_content = labels_dict[image_basename]
            else:
                # This should not happen if labels_dict was used for filtering
                raise ValueError(f"Label not found for {path}")

            # Write YOLO label file
            with open(os.path.join(lbl_dir, f"{image_id:06d}.txt"), "w") as f:
                f.write(label_content)

            image_id += 1
            n_sites += 1

        # Process nonsite patches (background, empty label files)
        for path in nonsites:
            try:
                img = read_and_preprocess(path)
            except Exception as e:
                print(f"Skipping {path}: {e}")
                continue

            fname = f"{image_id:06d}.jpg"
            Image.fromarray(img).save(os.path.join(img_dir, fname), quality=95)

            # Empty label file for background images
            with open(os.path.join(lbl_dir, f"{image_id:06d}.txt"), "w") as f:
                pass  # empty file

            image_id += 1
            n_bg += 1

        print(
            f"{split_name}: {image_id} images "
            f"({n_sites} sites, {n_bg} background)"
        )

    # Write data.yaml for Ultralytics
    abs_path = os.path.abspath(output_dir)
    data_yaml = {
        "path": abs_path,
        "train": "images/train",
        "val": "images/valid",
        "test": "images/test",
        "names": {0: "site"},
    }
    yaml_path = os.path.join(output_dir, "data.yaml")
    with open(yaml_path, "w") as f:
        yaml.dump(data_yaml, f, default_flow_style=False)

    print(f"\nDataset saved to: {output_dir}/")
    print(f"data.yaml: {yaml_path}")
    return yaml_path


def setup_external_testset(test_url, output_dir=None, labels_dict=None):
    """
    Download external test images from a Dropbox zip and populate the YOLO test split.

    Images are preprocessed and saved as JPGs. Labels are matched from labels_dict
    when available; images without labels get empty label files (treated as background).
    """
    if output_dir is None:
        output_dir = "detection_dataset_external"
    test_extract_dir = "input/test_external"
    os.makedirs(test_extract_dir, exist_ok=True)

    # Download zip
    if 'dropbox.com' in test_url:
        download_url = test_url.replace('www.dropbox.com', 'dl.dropboxusercontent.com')
        download_url = download_url.replace('?dl=0', '?dl=1')
        if '?dl=1' not in download_url:
            download_url = download_url + '?dl=1'
    else:
        download_url = test_url

    zip_path = os.path.join(test_extract_dir, "test.zip")
    print(f"Downloading external test images...")
    urllib.request.urlretrieve(download_url, zip_path)

    extract_subdir = os.path.join(test_extract_dir, "extracted")
    os.makedirs(extract_subdir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_subdir)

    # Find all image files
    test_paths = []
    for root, dirs, files in os.walk(extract_subdir):
        for f in files:
            if f.lower().endswith(('.tif', '.tiff', '.png', '.jpg', '.jpeg')):
                test_paths.append(os.path.join(root, f))

    test_paths = sorted(test_paths)
    print(f"Found {len(test_paths)} test images")

    img_dir = os.path.join(output_dir, "images", "test")
    lbl_dir = os.path.join(output_dir, "labels", "test")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    n_labeled = 0
    n_unlabeled = 0

    for path in test_paths:
        try:
            img = read_and_preprocess(path)
        except Exception as e:
            print(f"Skipping {path}: {e}")
            continue

        image_basename = os.path.splitext(os.path.basename(path))[0]
        fname = f"{image_basename}.jpg" # Use original filename
        Image.fromarray(img).save(os.path.join(img_dir, fname), quality=95)

        # Match against labels_dict
        if labels_dict and image_basename in labels_dict:
            label_content = labels_dict[image_basename]
            n_labeled += 1
        else:
            label_content = ""  # no annotation — treated as background
            n_unlabeled += 1

        with open(os.path.join(lbl_dir, f"{image_basename}.txt"), "w") as f: # Use original filename for label
            f.write(label_content)

    print(f"External test set: {len(test_paths)} images ({n_labeled} with labels, {n_unlabeled} without labels)")

    # Write a standalone data.yaml for model.val() on the external set
    abs_output_dir = os.path.abspath(output_dir)
    ext_yaml = {
        "path": abs_output_dir,
        "train": "images/test",   # placeholder — unused
        "val":   "images/test",   # placeholder — unused
        "test":  "images/test",
        "names": {0: "site"},
    }
    ext_yaml_path = os.path.join(output_dir, "external_data.yaml")
    with open(ext_yaml_path, "w") as f:
        yaml.dump(ext_yaml, f, default_flow_style=False)
    print(f"External data.yaml written to: {ext_yaml_path}")
    return ext_yaml_path

In [18]:
# Download and load YOLO labels
labels_dict = None
if USE_RAW and 'INPUT_ZIP_URL_LABELS' in globals():
    print("Downloading YOLO labels...")
    labels_dict = download_and_extract_labels(INPUT_ZIP_URL_LABELS)
    print(f"Loaded {len(labels_dict)} label files from Dropbox")
else:
    print("No YOLO labels URL provided, will generate default centered bboxes for sites")

# Build dataset — all training data goes to train/val (no test split)
data_yaml_path = build_yolo_dataset(labels_dict=labels_dict, max_train_images=None)

# Download and set up external test set (goes to its own directory)
external_data_yaml_path = None
if 'INPUT_ZIP_URL_TEST' in globals():
    print("\nSetting up external test set...")
    external_data_yaml_path = setup_external_testset(INPUT_ZIP_URL_TEST, labels_dict=labels_dict)
else:
    print("\nNo external test set URL provided, skipping external evaluation")


Downloaded to: input/yolo_labels/labels.zip
Loaded 168 label files
Loaded 168 label files from Dropbox
Dataset split: Train=80%, Val=20%, Test=0%
Found 369 site patches, 1000 nonsite patches
Using labels_dict: Keeping 112 labeled sites, excluding 257 unlabeled sites
Capped nonsites to 112
train: 178 images (89 sites, 89 background)
valid: 46 images (23 sites, 23 background)

Dataset saved to: detection_dataset_yolo/
data.yaml: detection_dataset_yolo/data.yaml

Setting up external test set...
Found 238 test images
External test set: 238 images (115 with labels, 123 without labels)
External data.yaml written to: detection_dataset_external/external_data.yaml


## Verify Dataset

In [ ]:
import supervision as sv
import matplotlib.pyplot as plt

ds = sv.DetectionDataset.from_yolo(
    images_directory_path=f"{DATASET_DIR}/images/train",
    annotations_directory_path=f"{DATASET_DIR}/labels/train",
    data_yaml_path=f"{DATASET_DIR}/data.yaml",
)

print(f"Training set: {len(ds)} images, classes: {ds.classes}")

# Show a few samples with bounding boxes
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
bbox_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.5)

sample_count = 0
sample_limit = 8
for idx in range(min(100, len(ds))):
    _, image, detections = ds[idx]
    if len(detections) > 0:
        ax = axes.flat[sample_count]
        labels = [ds.classes[cid] for cid in detections.class_id]
        annotated = bbox_annotator.annotate(image.copy(), detections)
        annotated = label_annotator.annotate(annotated, detections, labels)
        ax.imshow(annotated)
        ax.axis("off")
        sample_count += 1
        if sample_count >= sample_limit:
            break

for ax in axes.flat[sample_count:]:
    ax.axis("off")

plt.suptitle("Training samples with bounding boxes", fontsize=14)
plt.tight_layout()
plt.show()

Training set: 178 images, classes: ['site']


## Train YOLOv12 + DINOv3

Integration types (from least to most compute-heavy):
- **`single`**: DINOv3 preprocessor enhances the input image before YOLOv12 (lightest, most stable)
- **`dualp0p3`**: Input preprocessing + DINO features injected at P3 backbone level
- **`triple`**: Input + P3 + P4 injection (maximum enhancement)

DINO backbone variants: `vits16` (fastest), `vitb16` (balanced), `vitl16` (most powerful)

**Recommendation for small datasets (<2K images): use `single` integration with `vitb16`.**

In [ ]:
# ── Model configuration ──────────────────────────────────────────────
YOLO_SIZE = "s"             # n (nano), s (small), m (medium), l (large), x (xlarge)
DINO_VARIANT = "vitl16"     # vits16, vitb16, vitl16
INTEGRATION = "single"      # single, dual, dualp0p3, triple

# Use DINOv3 pretrained on SAT-493M satellite imagery instead of LVD-142M web images.
# Only available for vitl16. Set False to use the default web-pretrained model.
USE_SAT_PRETRAINED = True

# ── Training hyperparameters ─────────────────────────────────────────
EPOCHS = 50
BATCH_SIZE = 8             # 16 for A100, 8 for V100, 4 for T4
LR = 5e-4
IMG_SIZE = 768              # YOLO default input resolution

In [ ]:
# --- Redirect DINOv3 to satellite-pretrained checkpoint (SAT-493M) ---
# The DINOV3-YOLOV12 repo hardcodes "facebook/dinov3-vitl16-pretrain-lvd1689m"
# in 3 internal mapping dicts. This patches AutoModel/AutoConfig.from_pretrained
# to transparently redirect to the SAT-493M variant when loading vitl16.
#
# Reentrant: running this cell multiple times (or toggling USE_SAT_PRETRAINED)
# is safe — originals are stored once and the patch is applied/removed as needed.
from transformers import AutoModel, AutoConfig

_LVD_ID = "facebook/dinov3-vitl16-pretrain-lvd1689m"
_SAT_ID = "facebook/dinov3-vitl16-pretrain-sat493m"

# Store the true originals exactly once (survive repeated cell runs).
if not hasattr(AutoModel, "_orig_from_pretrained"):
    AutoModel._orig_from_pretrained = AutoModel.from_pretrained.__func__
if not hasattr(AutoConfig, "_orig_from_pretrained"):
    AutoConfig._orig_from_pretrained = AutoConfig.from_pretrained.__func__

if USE_SAT_PRETRAINED and DINO_VARIANT == "vitl16":
    @classmethod
    def _sat_model_fp(cls, name, *a, **kw):
        if name == _LVD_ID:
            name = _SAT_ID
        return cls._orig_from_pretrained(cls, name, *a, **kw)

    @classmethod
    def _sat_config_fp(cls, name, *a, **kw):
        if name == _LVD_ID:
            name = _SAT_ID
        return cls._orig_from_pretrained(cls, name, *a, **kw)

    AutoModel.from_pretrained = _sat_model_fp
    AutoConfig.from_pretrained = _sat_config_fp
    print(f"DINOv3 backbone redirected: {_LVD_ID} -> {_SAT_ID}")
else:
    # Restore originals so switching USE_SAT_PRETRAINED=False also works cleanly.
    AutoModel.from_pretrained = classmethod(AutoModel._orig_from_pretrained)
    AutoConfig.from_pretrained = classmethod(AutoConfig._orig_from_pretrained)
    print(f"Using default DINOv3 pretrained model for {DINO_VARIANT}")


In [ ]:
from ultralytics import YOLO
import ultralytics, pathlib

# Build the model config name following the repo's naming convention
if INTEGRATION == "single":
    config_name = f"yolov12{YOLO_SIZE}-dino3-{DINO_VARIANT}-single.yaml"
elif INTEGRATION == "dual":
    config_name = f"yolov12{YOLO_SIZE}-dino3-{DINO_VARIANT}-dual.yaml"
elif INTEGRATION == "dualp0p3":
    config_name = f"yolov12{YOLO_SIZE}-dino3-{DINO_VARIANT}-dualp0p3.yaml"
elif INTEGRATION == "triple":
    config_name = f"yolov12{YOLO_SIZE}-triple-dino3-{DINO_VARIANT}.yaml"
else:
    raise ValueError(f"Unknown integration type: {INTEGRATION}")

# If the YAML config doesn't exist (e.g. yolov12s-triple-dino3-vitl16), create it
# by adapting the vitb16 template with the correct DINO variant name.
cfg_dir = pathlib.Path(ultralytics.__file__).parent / "cfg" / "models" / "v12"
cfg_path = cfg_dir / config_name
if not cfg_path.exists():
    # Find a template: same YOLO size + integration, different DINO variant
    templates = sorted(cfg_dir.glob(f"yolov12{YOLO_SIZE}-triple-dino3-*.yaml"))
    if not templates:
        templates = sorted(cfg_dir.glob(f"yolov12*-triple-dino3-{DINO_VARIANT}.yaml"))
    if not templates:
        raise FileNotFoundError(f"No template found to generate {config_name}")
    template = templates[0]
    print(f"Creating {config_name} from template {template.name}")
    text = template.read_text()
    # Replace DINO variant references (vitb16->vitl16, CUSTOM_DINO_INPUT->dinov3_vitl16, etc.)
    for old_variant in ["CUSTOM_DINO_INPUT", "dinov3_vitb16", "dinov3_vits16"]:
        text = text.replace(old_variant, f"dinov3_{DINO_VARIANT}")
    cfg_path.write_text(text)
    print(f"Wrote {cfg_path}")

print(f"Model config: {config_name}")
model = YOLO(config_name)
print(model.info())

In [ ]:
# --- Fix PyTorch device mismatch bug in F.interpolate bilinear decomposition ---
# Recent PyTorch versions route F.interpolate(mode='bilinear') through a decomposed
# implementation (_upsample_linear) where shape-derived values stay on CPU while
# computation tensors are on CUDA, causing: "max is on cpu, different from cuda:0".
# This patches DINO3Preprocessor.forward and deterministic_interpolate to use
# F.grid_sample-based bilinear resize which avoids that code path entirely.

import torch
import torch.nn.functional as F
from ultralytics.nn.modules import block as _block


def _safe_bilinear_resize(x, target_h, target_w):
    """Bilinear resize using grid_sample — avoids PyTorch _upsample_linear decomposition bug."""
    B, C, H, W = x.shape
    target_h, target_w = int(target_h), int(target_w)
    if H == target_h and W == target_w:
        return x
    theta = torch.tensor([[1, 0, 0], [0, 1, 0]], dtype=x.dtype, device=x.device)
    theta = theta.unsqueeze(0).expand(B, -1, -1)
    grid = F.affine_grid(theta, [B, C, target_h, target_w], align_corners=False)
    return F.grid_sample(x, grid, mode='bilinear', align_corners=False, padding_mode='border')


def _patched_dino3_preprocessor_forward(self, x):
    """Patched forward that replaces F.interpolate with grid_sample-based resize."""
    batch_size, channels, height, width = x.shape
    original_input = x

    try:
        with torch.set_grad_enabled(not self.freeze_backbone):
            outputs = self.dino_model(x)
            if hasattr(outputs, "last_hidden_state"):
                dino_features = outputs.last_hidden_state

                patch_size = 16
                num_patches_h = height // patch_size
                num_patches_w = width // patch_size
                expected_patches = num_patches_h * num_patches_w

                dino_features = dino_features[:, 1:, :]
                dino_features = dino_features[:, :expected_patches, :]
                dino_features = dino_features.reshape(batch_size, num_patches_h, num_patches_w, -1)
                dino_features = dino_features.permute(0, 3, 1, 2)

                enhanced_features = self.feature_processor(dino_features)

                # Use grid_sample instead of F.interpolate to avoid decomposition bug
                enhanced_features = _safe_bilinear_resize(enhanced_features, height, width)

                enhanced_features = (enhanced_features + 1) / 2

                # Ensure residual_weight is on the correct device
                residual_weight = self.residual_weight.to(x.device)
                enhanced_image = (
                    original_input * (1 - residual_weight) + enhanced_features * residual_weight
                )

                return enhanced_image.clamp_(0.0, 1.0)
            else:
                return original_input

    except Exception as e:
        import traceback
        print(f"DINO3Preprocessor forward pass failed: {e}")
        traceback.print_exc()
        return original_input


# Also patch deterministic_interpolate used by DINO3Backbone
_orig_deterministic_interpolate = _block.deterministic_interpolate

def _patched_deterministic_interpolate(input, size=None, scale_factor=None, mode='bilinear', align_corners=False):
    if mode == 'bilinear' and size is not None and input.dim() == 4:
        return _safe_bilinear_resize(input, int(size[0]), int(size[1]))
    return _orig_deterministic_interpolate(input, size=size, scale_factor=scale_factor,
                                           mode=mode, align_corners=align_corners)


_block.DINO3Preprocessor.forward = _patched_dino3_preprocessor_forward
_block.deterministic_interpolate = _patched_deterministic_interpolate

In [ ]:
results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    lr0=LR,
    device=0,
    workers=4,
    patience=10,            # early stopping patience
    save=True,
    save_period=5,          # save checkpoint every N epochs
    plots=True,             # generate training plots
    verbose=True,
)

## Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Ultralytics saves training metrics to results.csv
results_csv = os.path.join(results.save_dir, "results.csv")
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot losses
    loss_cols = [c for c in df.columns if 'loss' in c.lower()]
    for c in loss_cols:
        axes[0].plot(df['epoch'], df[c], label=c)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training / Validation Loss")
    axes[0].legend(fontsize=8)

    # Plot mAP metrics
    map_cols = [c for c in df.columns if 'map' in c.lower() or 'precision' in c.lower() or 'recall' in c.lower()]
    for c in map_cols:
        axes[1].plot(df['epoch'], df[c], label=c)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Metric")
    axes[1].set_title("Validation mAP / Precision / Recall")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()
else:
    print(f"results.csv not found at {results_csv}")

## Evaluate on Test Set

In [ ]:
# ── External Test Set Evaluation Configuration ───────────────────────
# Controls whether to run evaluation on external test set in addition to the main test set
EXTERNAL_TEST = True

CONF_THRESHOLD = 0.02
IOU_THRESHOLD = 0.5

In [ ]:
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import supervision as sv
from PIL import Image as PILImage
import numpy as np

In [ ]:


# Load best weights
best_weights = os.path.join(results.save_dir, "weights", "best.pt")
model = YOLO(best_weights)
print(f"Loaded best weights from: {best_weights}")

# Run validation on test split
test_metrics = model.val(
    data=data_yaml_path,
    split="test",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD, # Added NMS IOU threshold
    verbose=True,
)

print(f"\nTest mAP@50:    {test_metrics.box.map50:.4f}")
print(f"Test mAP@50:95: {test_metrics.box.map:.4f}")


In [ ]:

test_img_dir = os.path.join(DATASET_DIR, "images", "test")
test_images_paths = sorted(glob.glob(os.path.join(test_img_dir, "*.jpg")))

# Find images that have ground-truth annotations
test_lbl_dir = os.path.join(DATASET_DIR, "labels", "test")
annotated_paths = []
for img_path in test_images_paths:
    lbl_path = os.path.join(test_lbl_dir, os.path.basename(img_path).replace(".jpg", ".txt"))
    if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
        annotated_paths.append(img_path)

num_annotated_images = len(annotated_paths)
num_cols = 5 # Number of columns for the subplot grid
num_rows = (num_annotated_images + num_cols - 1) // num_cols # Calculate required rows

# Adjust figsize based on the number of rows and columns
fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 5, num_rows * 5))
bbox_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.5)


# Initialize counters for detections
images_with_detections = 0
filenames_with_detections = []

for i, img_path in tqdm(enumerate(annotated_paths), total=num_annotated_images, desc="Processing annotated images"):
    # Flatten the axes array for easier indexing
    ax = axes.flat[i]
    image = cv.cvtColor(cv.imread(img_path), cv.COLOR_BGR2RGB)

    preds = model.predict(img_path, conf=CONF_THRESHOLD, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(preds)

    labels = [f"site {c:.2f}" for c in detections.confidence] if detections.confidence is not None and len(detections) > 0 else []
    annotated = bbox_annotator.annotate(image.copy(), detections)
    annotated = label_annotator.annotate(annotated, detections, labels)
    ax.imshow(annotated)
    ax.axis("off")

    # Check for detections and update counters
    if len(detections) > 0:
        images_with_detections += 1
        filenames_with_detections.append(os.path.basename(img_path))


# Turn off any remaining empty subplots
for ax in axes.flat[num_annotated_images:]:
    ax.axis("off")

plt.suptitle("Test set predictions (YOLOv12+DINOv3)", fontsize=14)
plt.tight_layout()
plt.show()

# Output the count and filenames at the end
print(f"\nTotal annotated images with YOLO detections: {images_with_detections}")
print("Filenames with detections:")
for filename in filenames_with_detections:
    print(filename)

## External Test Set Evaluation

Evaluate the model on an external test dataset (if `EXTERNAL_TEST=True`). This allows testing the model on completely independent data from different sources or regions.

In [ ]:
EXTERNAL_TEST=True

In [ ]:
if EXTERNAL_TEST:
    if external_data_yaml_path is None:
        print("No external test set available (INPUT_ZIP_URL_TEST was not set). Skipping.")
    else:
        external_test_dir = os.path.join(os.path.dirname(external_data_yaml_path), "images", "test")
        if not os.path.exists(external_test_dir) or len(os.listdir(external_test_dir)) == 0:
            print(f"External test directory empty or missing: {external_test_dir}")
        else:
            print("=" * 70)
            print("RUNNING EXTERNAL TEST SET EVALUATION")
            print("=" * 70)

            external_test_metrics = model.val(
                data=external_data_yaml_path,
                split="test",
                imgsz=IMG_SIZE,
                batch=BATCH_SIZE,
                conf=CONF_THRESHOLD,
                iou=IOU_THRESHOLD, # Added NMS IOU threshold
                verbose=True,
            )

            print("\n" + "=" * 70)
            print("EXTERNAL TEST SET RESULTS")
            print("=" * 70)
            print(f"External Test mAP@50:    {external_test_metrics.box.map50:.4f}")
            print(f"External Test mAP@50:95: {external_test_metrics.box.map:.4f}")

            precision_val = external_test_metrics.box.p[0] if external_test_metrics.box.p else 0.0
            recall_val = external_test_metrics.box.r[0] if external_test_metrics.box.r else 0.0

            print(f"External Test Precision: {precision_val:.4f}")
            print(f"External Test Recall:    {recall_val:.4f}")
            print("=" * 70)


In [ ]:
if EXTERNAL_TEST:
    if external_data_yaml_path is None:
        print("No external test set available. Skipping visualisation.")
    else:
        print("Visualizing external test set predictions...")

        external_test_img_dir = os.path.join(os.path.dirname(external_data_yaml_path), "images", "test")
        external_test_lbl_dir = os.path.join(os.path.dirname(external_data_yaml_path), "labels", "test")
        external_test_images_paths = sorted(glob.glob(os.path.join(external_test_img_dir, "*.jpg")))

        if len(external_test_images_paths) == 0:
            print(f"No images found in {external_test_img_dir}")
        else:
            # Find images that have ground-truth annotations
            annotated_paths = []
            for img_path in external_test_images_paths:
                lbl_path = os.path.join(external_test_lbl_dir, os.path.basename(img_path).replace(".jpg", ".txt"))
                if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
                    annotated_paths.append(img_path)

            # Limit to first 20 images for visualisation
            #annotated_paths = annotated_paths[:20]
            num_annotated_images = len(annotated_paths)

            if num_annotated_images == 0:
                print("No annotated external test images found for visualisation.")
            else:
                num_cols = 5
                num_rows = (num_annotated_images + num_cols - 1) // num_cols

                fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 5, num_rows * 5))
                bbox_annotator = sv.BoxAnnotator(thickness=2)
                label_annotator = sv.LabelAnnotator(text_scale=0.5)

                images_with_detections = 0
                filenames_with_detections = []

                for i, img_path in tqdm(enumerate(annotated_paths), total=num_annotated_images, desc="Processing external test images"):
                    ax = axes.flat[i] if num_annotated_images > 1 else axes
                    image = cv.cvtColor(cv.imread(img_path), cv.COLOR_BGR2RGB)

                    preds = model.predict(img_path, conf=CONF_THRESHOLD, verbose=False)[0]
                    detections = sv.Detections.from_ultralytics(preds)

                    labels = [f"site {c:.2f}" for c in detections.confidence] if detections.confidence is not None and len(detections) > 0 else []
                    annotated = bbox_annotator.annotate(image.copy(), detections)
                    annotated = label_annotator.annotate(annotated, detections, labels)
                    ax.imshow(annotated)
                    ax.axis("off")

                    if len(detections) > 0:
                        images_with_detections += 1
                        filenames_with_detections.append(os.path.basename(img_path))

                if num_annotated_images > 1:
                    for ax in axes.flat[num_annotated_images:]:
                        ax.axis("off")

                plt.suptitle("External Test Set Predictions (YOLOv12+DINOv3)", fontsize=14)
                plt.tight_layout()
                plt.show()

                print(f"\nTotal external test images with detections: {images_with_detections}/{num_annotated_images}")
                print("Filenames with detections:")
                for filename in filenames_with_detections[:10]:
                    print(f"  - {filename}")
                if len(filenames_with_detections) > 10:
                    print(f"  ... and {len(filenames_with_detections) - 10} more")


In [ ]:
#HALT

## Inference on Large Tiles

Sliding window approach:
1. Slide a window (e.g., 768×768) across a large GeoTIFF tile
2. Run YOLOv12+DINOv3 on each window
3. Merge overlapping detections with NMS
4. Export results as GeoJSON with geo-coordinates

In [ ]:
import os
import urllib.request
import numpy as np
import pandas as pd
import rasterio
from PIL import Image
from tqdm import tqdm
from typing import Tuple, List

try:
    import rvt.vis
except ImportError:
    print("Warning: rvt-py not installed. LiDAR-to-hillshade conversion will not be available.")

In [ ]:
def download_tile(dropbox_url: str, output_dir: str) -> str:
    """Download a tile from Dropbox URL."""
    original_filename = dropbox_url.split('/')[-1].split('?')[0]
    final_output_path = os.path.join(output_dir, original_filename)

    if os.path.exists(final_output_path):
        print(f"File already exists: {final_output_path}. Skipping download.")
        return final_output_path

    if 'dropbox.com' in dropbox_url:
        download_url = dropbox_url.replace('www.dropbox.com', 'dl.dropboxusercontent.com')
        download_url = download_url.replace('?dl=0', '?dl=1')
        if '?dl=1' not in download_url:
            download_url = download_url + '?dl=1'
    else:
        download_url = dropbox_url

    print(f"Downloading tile from: {dropbox_url}")
    os.makedirs(output_dir, exist_ok=True)
    urllib.request.urlretrieve(download_url, final_output_path)
    print(f"Downloaded to: {final_output_path}")
    return final_output_path


def lidar_to_hillshade(lidar_path: str, output_path: str = None) -> Tuple[np.ndarray, dict]:
    """Convert LiDAR elevation data to hillshade using RVT."""
    print(f"Converting LiDAR to hillshade: {lidar_path}")
    if output_path:
        hillshade_filename = os.path.basename(lidar_path).replace('.tif', '_hillshade.tif')
        output_path = os.path.join(os.path.dirname(output_path), hillshade_filename)

    with rasterio.open(lidar_path) as src:
        elevation = src.read(1)
        profile = src.profile.copy()
        transform = src.transform
        crs = src.crs
        resolution = abs(transform[0])

    hillshade = rvt.vis.hillshade(
        dem=elevation, resolution_x=resolution, resolution_y=resolution,
        sun_azimuth=315, sun_elevation=35,
    )
    hs = ((hillshade - hillshade.min()) / (hillshade.max() - hillshade.min()) * 255).astype(np.uint8)

    if output_path:
        profile.update(dtype=rasterio.uint8, count=1, nodata=0)
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(hs, 1)
        print(f"Hillshade saved to: {output_path}")

    metadata = {
        'transform': transform, 'crs': crs,
        'width': hs.shape[1], 'height': hs.shape[0], 'resolution': resolution,
    }
    return hs, metadata


def read_image_tile(image_path: str) -> Tuple[np.ndarray, dict]:
    """Read a GeoTIFF image tile directly."""
    print(f"Reading image tile: {image_path}")
    with rasterio.open(image_path) as src:
        num_bands = src.count
        transform = src.transform
        crs = src.crs
        resolution = abs(transform[0])
        if num_bands >= 3:
            r, g, b = src.read(1), src.read(2), src.read(3)
            image = np.stack([r, g, b], axis=-1)
        else:
            image = src.read(1)

    if image.dtype != np.uint8:
        img_min, img_max = image.min(), image.max()
        if img_max > img_min:
            image = ((image - img_min) / (img_max - img_min) * 255).astype(np.uint8)
        else:
            image = np.zeros_like(image, dtype=np.uint8)

    metadata = {
        'transform': transform, 'crs': crs,
        'width': image.shape[1], 'height': image.shape[0], 'resolution': resolution,
    }
    bands_str = f"{num_bands} bands (RGB)" if num_bands >= 3 else f"{num_bands} band (grayscale)"
    print(f"Image size: {image.shape[1]}x{image.shape[0]}, {bands_str}")
    return image, metadata

In [ ]:
import supervision as sv


def sliding_window_detection(
    image_array: np.ndarray,
    model,
    metadata: dict,
    input_type: str = 'satellite',
    window_size: int = 768,
    stride: int = None,
    conf_threshold: float = 0.3,
    nms_iou_threshold: float = 0.5,
    fraction: float = 1.0,
) -> pd.DataFrame:
    """
    Sliding window object detection on a large tile using YOLOv12+DINOv3.
    """
    if stride is None:
        stride = int(window_size * 0.75)  # 25% overlap

    # Crop if fraction < 1
    if fraction < 1.0:
        h, w = image_array.shape[:2]
        crop_h, crop_w = int(h * fraction), int(w * fraction)
        image_array = image_array[:crop_h, :crop_w] if image_array.ndim == 2 else image_array[:crop_h, :crop_w, :]
        print(f"Cropped to {fraction*100:.0f}%: {crop_w}x{crop_h}")

    is_rgb = image_array.ndim == 3
    height, width = image_array.shape[:2]

    n_x = max(1, (width - window_size) // stride + 1)
    n_y = max(1, (height - window_size) // stride + 1)
    total = n_x * n_y
    print(f"Tile: {width}x{height}, windows: {n_x}x{n_y} = {total}, stride: {stride}")

    all_boxes = []
    all_confs = []

    preprocess_fn = get_preprocessing_function()

    for y in tqdm(range(0, height - window_size + 1, stride), desc="Rows"):
        for x in range(0, width - window_size + 1, stride):
            if is_rgb:
                window = image_array[y:y + window_size, x:x + window_size, :]
            else:
                window = image_array[y:y + window_size, x:x + window_size]
                if input_type != 'lidar':
                    window = np.stack([window] * 3, axis=-1)
            window = preprocess_fn(window)

            # YOLOv12 predict accepts numpy arrays directly
            preds = model.predict(window, conf=conf_threshold, verbose=False)[0]

            if len(preds.boxes) > 0:
                for bbox, conf in zip(preds.boxes.xyxy.cpu().numpy(), preds.boxes.conf.cpu().numpy()):
                    x1, y1, x2, y2 = bbox
                    all_boxes.append([x1 + x, y1 + y, x2 + x, y2 + y])
                    all_confs.append(float(conf))

    if not all_boxes:
        print("No detections found.")
        return pd.DataFrame(columns=[
            'x1', 'y1', 'x2', 'y2', 'confidence',
            'center_x', 'center_y', 'lon', 'lat',
        ])

    # Merge overlapping detections with NMS
    boxes_array = np.array(all_boxes)
    confs_array = np.array(all_confs)
    class_ids = np.zeros(len(all_boxes), dtype=int)

    merged = sv.Detections(
        xyxy=boxes_array,
        confidence=confs_array,
        class_id=class_ids,
    )
    merged = merged.with_nms(threshold=nms_iou_threshold)
    print(f"Detections before NMS: {len(all_boxes)}, after NMS: {len(merged)}")

    # Convert to DataFrame with geo-coordinates
    transform = metadata['transform']
    results = []
    for bbox, conf in zip(merged.xyxy, merged.confidence):
        x1, y1, x2, y2 = bbox
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        # Use the pixel center when converting to spatial coordinates so
        # cx/cy map to the center of the detection in CRS space.
        lon, lat = rasterio.transform.xy(transform, cy, cx, offset='center')
        results.append({
            'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
            'confidence': conf,
            'center_x': cx, 'center_y': cy,
            'lon': lon, 'lat': lat,
        })

    df = pd.DataFrame(results)
    print(f"\nTotal detections: {len(df)}")
    print(f"Mean confidence: {df['confidence'].mean():.4f}")
    print(f"High confidence (>0.7): {(df['confidence'] > 0.7).sum()}")
    return df

In [ ]:
import json as _json


def detections_to_geojson(df: pd.DataFrame, rasterio_transform, crs=None, output_path: str = None) -> dict:
    """
    Convert detection results to GeoJSON with bounding box polygons.
    """
    from rasterio.crs import CRS as _CRS

    need_reproject = False
    if crs is not None:
        src_crs = _CRS(crs)
        if src_crs.to_epsg() != 4326:
            need_reproject = True
            dst_crs = _CRS.from_epsg(4326)

    features = []
    for _, row in df.iterrows():
        x1, y1, x2, y2 = int(row['x1']), int(row['y1']), int(row['x2']), int(row['y2'])
        cols = [x1, x2, x2, x1, x1]
        rows = [y1, y1, y2, y2, y1]
        xs, ys = rasterio.transform.xy(rasterio_transform, rows, cols, offset='ul')

        if need_reproject:
            from rasterio.warp import transform as _warp
            xs, ys = _warp(src_crs, dst_crs, list(xs), list(ys))

        coordinates = [[[float(x), float(y)] for x, y in zip(xs, ys)]]

        features.append({
            "type": "Feature",
            "geometry": {"type": "Polygon", "coordinates": coordinates},
            "properties": {
                "confidence": round(float(row['confidence']), 6),
                "label": "site",
            },
        })

    geojson = {"type": "FeatureCollection", "features": features}

    if output_path:
        with open(output_path, 'w') as f:
            _json.dump(geojson, f)
        print(f"GeoJSON saved to: {output_path} ({len(features)} features)")

    return geojson

In [ ]:
def run_detection_pipeline(
    dropbox_url: str,
    model,
    output_csv: str = "detection_results.csv",
    output_geojson: str = "detection_results.geojson",
    window_size: int = 768,
    stride: int = None,
    conf_threshold: float = 0.1,
    nms_iou_threshold: float = 0.5,
    input_type: str = 'satellite',
    save_hillshade: bool = True,
    temp_dir: str = "./inference_temp",
    fraction: float = 1.0,
) -> pd.DataFrame:
    """
    Complete detection pipeline: download tile, run YOLOv12+DINOv3, save results.
    """
    os.makedirs(temp_dir, exist_ok=True)

    # Step 1: Download tile
    print("\n" + "=" * 70)
    print("STEP 1: Downloading tile")
    print("=" * 70)
    downloaded_path = download_tile(dropbox_url, temp_dir)

    # Step 2: Prepare image
    print("\n" + "=" * 70)
    if input_type == 'lidar':
        print("STEP 2: Converting LiDAR elevation to hillshade")
        print("=" * 70)
        hs_path = os.path.join(temp_dir, "hillshade.tif") if save_hillshade else None
        image_array, metadata = lidar_to_hillshade(downloaded_path, hs_path)
    else:
        print(f"STEP 2: Reading {input_type} image tile")
        print("=" * 70)
        image_array, metadata = read_image_tile(downloaded_path)

    # Step 3: Run detection
    print("\n" + "=" * 70)
    print("STEP 3: Running sliding window detection with YOLOv12+DINOv3")
    print("=" * 70)
    results_df = sliding_window_detection(
        image_array=image_array,
        model=model,
        metadata=metadata,
        input_type=input_type,
        window_size=window_size,
        stride=stride,
        conf_threshold=conf_threshold,
        nms_iou_threshold=nms_iou_threshold,
        fraction=fraction,
    )

    # Step 4: Save results
    print("\n" + "=" * 70)
    print("STEP 4: Saving results")
    print("=" * 70)
    if len(results_df) > 0:
        results_df.to_csv(output_csv, index=False)
        print(f"CSV saved to: {output_csv}")

        if output_geojson:
            detections_to_geojson(
                results_df,
                rasterio_transform=metadata['transform'],
                crs=metadata.get('crs'),
                output_path=output_geojson,
            )
    else:
        print("No detections to save.")

    # Summary
    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)
    print(f"Input type: {input_type}")
    if fraction < 1.0:
        print(f"Tile fraction: {fraction*100:.0f}%")
    print(f"Total detections: {len(results_df)}")
    if len(results_df) > 0:
        print(f"Mean confidence: {results_df['confidence'].mean():.4f}")
        print(f"High confidence (>0.7): {(results_df['confidence'] > 0.7).sum()}")

    return results_df

## Configure Inference Tile

In [ ]:
# === Configuration: choose input type ===
# 'lidar'     = elevation GeoTIFF -> converted to hillshade via RVT
# 'satellite' = RGB satellite GeoTIFF -> fed directly to the model
INPUT_TYPE = 'satellite'

# --- LiDAR tiles ---
LIDAR_TILE_URL = "https://www.dropbox.com/scl/fi/s345nxlz1zynur4zx8usu/test786.tif?rlkey=0zvvni37tk338bbjof6zp8zx9&st=t56siocp&dl=0"

# --- Satellite tiles ---
SATELLITE_TILE_URL = "https://www.dropbox.com/scl/fi/yljarvosujnb6to7ih1i8/area_49.097560_10.527880_to_49.029980_10.778630_z18.tif?rlkey=u3juj2huyhkiqy1mnmzmjpapm&dl=0"

TILE_URL = SATELLITE_TILE_URL if INPUT_TYPE == 'satellite' else LIDAR_TILE_URL

In [ ]:
results_df = run_detection_pipeline(
    dropbox_url=TILE_URL,
    model=model,
    output_csv="detection_results.csv",
    output_geojson="detection_results.geojson",
    window_size=768,
    conf_threshold=0.11,
    nms_iou_threshold=0.5,
    input_type=INPUT_TYPE,
    fraction=1.0,
)

## Visualize Detections on Tile

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Rectangle


def visualize_tile_detections(
    tile_url: str,
    results_df: pd.DataFrame,
    input_type: str = 'satellite',
    temp_dir: str = "./inference_temp",
    scale: float = 0.1,
    min_confidence: float = 0.3,
):
    """Visualize detection bounding boxes overlaid on the tile."""
    # Load the tile
    tile_path = download_tile(tile_url, temp_dir)
    if input_type == 'lidar':
        image_array, _ = lidar_to_hillshade(tile_path)
        if image_array.ndim == 2:
            image_array = np.stack([image_array] * 3, axis=-1)
    else:
        image_array, _ = read_image_tile(tile_path)

    # Resize for display
    h, w = image_array.shape[:2]
    new_w, new_h = int(w * scale), int(h * scale)
    display_img = cv.resize(image_array, (new_w, new_h))

    fig, ax = plt.subplots(1, 1, figsize=(16, 16))
    ax.imshow(display_img)

    # Filter by confidence
    df = results_df[results_df['confidence'] >= min_confidence]

    for _, row in df.iterrows():
        x1 = row['x1'] * scale
        y1 = row['y1'] * scale
        bw = (row['x2'] - row['x1']) * scale
        bh = (row['y2'] - row['y1']) * scale
        conf = row['confidence']

        color = 'red' if conf > 0.7 else 'orange' if conf > 0.5 else 'yellow'
        rect = Rectangle((x1, y1), bw, bh, linewidth=1.5, edgecolor=color, facecolor='none')
        ax.add_patch(rect)

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='none', edgecolor='red', label=f'High conf (>0.7): {(df["confidence"] > 0.7).sum()}'),
        Patch(facecolor='none', edgecolor='orange', label=f'Medium conf (0.5-0.7): {((df["confidence"] > 0.5) & (df["confidence"] <= 0.7)).sum()}'),
        Patch(facecolor='none', edgecolor='yellow', label=f'Low conf (0.3-0.5): {((df["confidence"] > 0.3) & (df["confidence"] <= 0.5)).sum()}'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
    ax.set_title(f"YOLOv12+DINOv3 Detections: {len(df)} sites (min conf: {min_confidence})")
    ax.axis('off')
    plt.tight_layout()
    plt.show()


visualize_tile_detections(
    tile_url=TILE_URL,
    results_df=results_df,
    input_type=INPUT_TYPE,
    min_confidence=0.1,
)

## Download Results

In [ ]:
# Download the results files (Google Colab)
try:
    from google.colab import files
    print("Downloading detection_results.geojson and detection_results.csv...")
    files.download('detection_results.geojson')
    #files.download('detection_results.csv')
except ImportError:
    print("Not running in Colab. Results saved to:")
    print("  - detection_results.csv")
    print("  - detection_results.geojson")